In [8]:
import os
import sys
import re

import numpy as np
import pandas as pd

from pathlib import Path

In [18]:
sys.path.insert(
    0,
    str(Path("../../lift-analyzer-tool").resolve())
)

In [24]:
import src as liftanalyzer

In [19]:
EXERCISE = "deadlift"
# put the file in the data folder
ESTIMATIONS_FILE = "deadlift-estimations-rpe-marked.csv"


In [20]:
csv_path = os.getcwd() / Path("data/" + ESTIMATIONS_FILE)
df = pd.read_csv(csv_path)

In [21]:
df.sample(10)

,filename,frame,time_s,weight,height,rpe,detected,img0_x,img0_y,img0_z,...,world29_z,world30_x,world30_y,world30_z,world31_x,world31_y,world31_z,world32_x,world32_y,world32_z
2040,thor-501kg_rpe9_204cm.mp4,8,0.2983,501.0,204.0,9,1,0.500480,0.538566,-0.662342,...,-0.047977,-0.255683,0.532225,-0.093844,0.245220,0.582721,-0.123020,-0.327447,0.488036,-0.199945
6892,singleton_198cm_420kg_rpe7.mp4,37,1.2507,420.0,198.0,7,1,0.479527,0.254207,-0.314961,...,0.069181,-0.205664,0.730320,0.043826,0.270943,0.768183,-0.011109,-0.244530,0.736770,-0.052204
5095,shaw500training2_200cm_rpe8.mp4,2,0.0674,NaN,200.0,8,1,0.466733,0.461094,-0.601939,...,-0.014415,-0.276983,0.472099,0.009894,0.241975,0.585318,-0.082804,-0.333068,0.498127,-0.072506
8731,elbel_370kg_rpe7.mp4,58,1.9720,370.0,NaN,7,1,0.502442,0.179085,-0.149760,...,0.010257,-0.131225,0.786906,0.050396,0.140840,0.813580,-0.065757,-0.217910,0.791158,-0.016203
1584,deadlift_6_rpe5.mp4,59,1.9686,NaN,NaN,5,1,0.514993,0.142114,0.058212,...,0.334323,0.081986,0.790221,0.039701,-0.057072,0.739811,0.387609,0.176039,0.766991,0.042338
6022,eddie_williams_400kg_193cm_rpe8.mp4,120,4.0449,400.0,193.0,8,1,0.492009,0.264100,-0.358847,...,-0.011914,-0.080811,0.740193,0.013319,0.174430,0.762027,-0.085771,-0.134486,0.730789,-0.078606
1745,thor-120kg_rpe5_204cm.mp4,3,0.1136,120.0,204.0,5,1,0.599097,0.414407,-0.653970,...,-0.106476,-0.132284,0.443066,0.281247,0.424624,0.559480,-0.117768,-0.073731,0.482388,0.254811
7238,g_bilton_440kg_200cm_rpe9.mp4,138,4.6920,440.0,200.0,9,1,0.503408,0.239923,-0.304072,...,0.020196,-0.206521,0.612139,0.014412,0.225300,0.680420,-0.031218,-0.240134,0.622218,-0.063215
3789,shaw_car6_200cm_rpe7.mp4,0,0.0000,NaN,200.0,7,1,0.393433,0.337586,-0.702917,...,-0.085182,-0.221634,0.512487,-0.035568,0.189407,0.648233,-0.174836,-0.282438,0.517541,-0.129267
4038,shaw_car10_200cm_rpe8.mp4,34,1.1461,NaN,200.0,8,1,0.418079,0.086378,-0.474785,...,0.109666,-0.219075,0.702251,0.223935,0.188982,0.795413,0.035039,-0.281826,0.724600,0.160558


In [23]:
footage_samples = df.groupby("filename").groups.keys()
print(footage_samples)

dict_keys(['a_andrade_188cm_454kg_rpe8.mp4', 'a_andrade_188cm_470kg_rpe9.mp4', 'a_andrade_400kg_188cm_rpe7.mp4', 'a_andrade_420kg_188cm_rpe7.mp4', 'a_bishop_191cm_454kg_rpe8.mp4', 'a_bishop_191cm_470kg_rpe9.mp4', 'a_bishop_400kg_191cm_rpe6.mp4', 'b_johnson_200cm_400kg_rpe7.mp4', 'b_johnson_200cm_420kg_rpe8.mp4', 'b_johnson_200cm_440kg_rpe8.mp4', 'ben_donin_350kg_rpe7.mp4', 'ben_donin_390kg_rpe10.mp4', 'deadlift_10_rpe10.mp4', 'deadlift_11_conventional_rpe5.mp4', 'deadlift_11_sumo_rpe5.mp4', 'deadlift_12_conventional_rpe5.mp4', 'deadlift_12_sumo_rpe5.mp4', 'deadlift_13_rpe5.mp4', 'deadlift_14_rpe5.mp4', 'deadlift_15_rpe5.mp4', 'deadlift_16_rpe5.mp4', 'deadlift_17_rpe5.mp4', 'deadlift_18_rpe5.mp4', 'deadlift_19_rpe5.mp4', 'deadlift_1_rpe5.mp4', 'deadlift_20_rpe5.mp4', 'deadlift_21_1rpe5.mp4', 'deadlift_21_2rpe5.mp4', 'deadlift_22_rpe5.mp4', 'deadlift_23_rpe5.mp4', 'deadlift_24_rpe5.mp4', 'deadlift_26_1rpe5.mp4', 'deadlift_26_2rpe5.mp4', 'deadlift_27_1rpe5.mp4', 'deadlift_27_2rpe5.mp4', '

In [46]:
from types import SimpleNamespace as NS
import numpy as np

N_LANDMARKS = 33
fake_frame = np.zeros((720, 1280, 3), dtype=np.uint8)

def row_to_estimation(r):
    if not r["detected"]:
        return NS(pose_landmarks=[], pose_world_landmarks=[])
    img = [NS(x=r[f"img{i}_x"], y=r[f"img{i}_y"], z=r[f"img{i}_z"],
              visibility=1.0, presence=1.0) for i in range(N_LANDMARKS)]
    world = [NS(x=r[f"world{i}_x"], y=r[f"world{i}_y"], z=r[f"world{i}_z"],
                visibility=1.0, presence=1.0) for i in range(N_LANDMARKS)]
    return NS(pose_landmarks=[img], pose_world_landmarks=[world])



In [71]:
metrics_by_footage = {}

for footage_sample in footage_samples:
    try:
        footage_df = df[df.filename == footage_sample].sort_values("frame")
        height = footage_df.height.iloc[0] or 0.0
        rpe = footage_df.rpe.iloc[0]
        print(f"Estimating metrics for {footage_sample} [ height: {height}, rpe: {rpe}")

        metrics = liftanalyzer.metrics.mediapipe_metrics.MediapipeMetrics()
        if pd.notna(height) and height > 0:
            estimated_frames = [(fake_frame, row_to_estimation(r)) for _, r in footage_df.iterrows()]
            metrics.calibrate_scale(height / 100.0, estimated_frames)
        history = None

        for _, r in footage_df.iterrows():
            estimation = row_to_estimation(r)
            timestamp = int(r["time_s"] * 1e9) 
            history = metrics.get_metrics(fake_frame, estimation, timestamp)
        rep_ranges = liftanalyzer.utils.biomechanics.get_rep_ranges(EXERCISE, history)
        average_velocity = metrics.get_average_velocity(history, rep_ranges)
        peak_velocity = metrics.get_peak_velocity(history, rep_ranges)
        metrics_by_footage[footage_sample] = { "rpe": rpe, "average_velocity": average_velocity, "peak_velocity": peak_velocity }
    except Exception as e:
        print(e)


Estimating metrics for a_andrade_188cm_454kg_rpe8.mp4 [ height: 188.0, rpe: 8
calibrated: 200.6 px/m (H=1.88 m, n=2 segments)
Estimating metrics for a_andrade_188cm_470kg_rpe9.mp4 [ height: 188.0, rpe: 9
calibrated: 211.9 px/m (H=1.88 m, n=2 segments)
Estimating metrics for a_andrade_400kg_188cm_rpe7.mp4 [ height: 188.0, rpe: 7
calibrated: 203.2 px/m (H=1.88 m, n=2 segments)
Estimating metrics for a_andrade_420kg_188cm_rpe7.mp4 [ height: 188.0, rpe: 7
calibrated: 205.9 px/m (H=1.88 m, n=2 segments)
Estimating metrics for a_bishop_191cm_454kg_rpe8.mp4 [ height: 191.0, rpe: 8
calibrated: 207.2 px/m (H=1.91 m, n=2 segments)
Estimating metrics for a_bishop_191cm_470kg_rpe9.mp4 [ height: 191.0, rpe: 9
calibrated: 221.2 px/m (H=1.91 m, n=2 segments)
Estimating metrics for a_bishop_400kg_191cm_rpe6.mp4 [ height: 191.0, rpe: 6
calibrated: 210.1 px/m (H=1.91 m, n=2 segments)
Estimating metrics for b_johnson_200cm_400kg_rpe7.mp4 [ height: 200.0, rpe: 7
calibrated: 212.1 px/m (H=2.00 m, n=2 segme

In [79]:
velocity_rpe_df = pd.DataFrame(metrics_by_footage).T

In [93]:
out = velocity_rpe_df[["average_velocity", "peak_velocity", "rpe"]].copy()
out["rpe"] = out["rpe"].astype(int).astype(str).replace({"5": "<5"})
out

,average_velocity,peak_velocity,rpe
a_andrade_188cm_454kg_rpe8.mp4,0.229337,0.331736,8
a_andrade_188cm_470kg_rpe9.mp4,0.159738,0.308038,9
a_andrade_400kg_188cm_rpe7.mp4,0.346808,0.521046,7
a_andrade_420kg_188cm_rpe7.mp4,0.310702,0.464875,7
a_bishop_191cm_454kg_rpe8.mp4,0.219868,0.411419,8
...,...,...,...
thor_454kg_204cm_rpe9.mp4,0.130034,0.333431,9
thor_455kg_200cm_rpe10.mp4,0.123358,0.182729,10
thor_470kg_204cm_rpe8.mp4,0.285099,0.407794,8
vladimir_370kg_rpe8.mp4,0.051486,0.191305,8


In [92]:
out.to_csv("./data"index=False, header=False)

'0.22933693905128455,0.33173617094689617,8\n0.1597383208445731,0.3080382823389719,9\n0.3468075844748047,0.521045588161066,7\n0.3107023783835206,0.4648746695653772,7\n0.2198681825598272,0.41141891671373004,8\n0.2213100219921527,0.32040862233444806,9\n0.3767987233249024,0.5990793472812492,6\n0.3852710510101735,0.5661089451040257,7\n0.3206830362733487,0.45997706723092047,8\n0.203653436603936,0.35935308664823423,8\n0.11841565129565713,0.2975491118921823,7\n0.09101468596803629,0.21673933495283623,10\n0.2449062310267411,0.34238327557891374,10\n0.22358533524180327,0.3531642168644483,<5\n0.14174706849487645,0.20009178674411826,<5\n0.2798258016457283,0.37743345616375273,<5\n0.1746242609694457,0.2905446682109079,<5\n0.17527688155685775,0.30339157223392976,<5\n0.17321016452271484,0.328990227898475,<5\n0.5433721929091251,0.9798455554681866,<5\n0.15615379411856414,0.2023652420947852,<5\n0.2770320572861354,0.3828420396913101,<5\n0.1310359938627007,0.3180580241277655,<5\n0.16489489387237372,0.4129388